# [Bacterial Colony Growth](@id Bacteries)

In this example, we are going to create a bacterial model and grow a colony using it.

 - The implementation of the force interaction dynamics is the one presented by [Volfson et al. (2008)](https://www.pnas.org/doi/abs/10.1073/pnas.0706805105)
 - We use GPU accelerated dynamics

As described in other models, it is advised that the models are constructed by parts to avoid having to find bugs in a very complex system. Hence, we will split the model in two parts:

 - Forces model
 - Growth model

## Load the packages

In [92]:
import Pkg
Pkg.activate("../")
Pkg.instantiate()


  Activating project at `~/Escritorio/Projects/Pushing/ABM_CBM`


In [93]:
import Pkg; Pkg.add(["Revise", "GLMakie"])

   Resolving package versions...
  No Changes to `~/Escritorio/Projects/Pushing/ABM_CBM/Project.toml`
  No Changes to `~/Escritorio/Projects/Pushing/ABM_CBM/Manifest.toml`


In [94]:
using Revise
using CellBasedModels
using CUDA
using Distributions
using GLMakie
import GLMakie: Point3f, Cylinder, Sphere, NoShading #Can be changes to Cairo or CLMakie

In [95]:
model = ABM(
    2,

    agent = Dict(
        :neighbor => Bool,
    ),

    agentRule = quote

        @loopOverNeighbors i2 begin
            
            if id == 1

                neighbor[i2] = true

            end

        end

    end,

    neighborsAlg = CBMNeighbors.CellLinked(cellEdge = 4.0)

)

PARAMETERS
	x (Float64 agent)
	y (Float64 agent)
	neighbor (Bool agent)


UPDATE RULES
agentRule
 @loopOverNeighbors i2 if id == 1
        neighbor__[i2] = true
    end



In [96]:
com = Community(model, N=2000, simBox=[0. 31.; 0. 21.])

com.neighbor = false

com.x = rand(Uniform(0, 31), com.N)
com.y = rand(Uniform(0, 31), com.N)

loadToPlatform!(com)
step!(com)

In [97]:
fig = Figure(size = (600, 600))
ax = Axis(fig[1, 1], xlabel = "x", ylabel = "y", title = "Neighbors")
fig[1, 1] = ax

scatter!(ax, com.x, com.y, color = com.neighbor, markersize = 8, colormap = :viridis)

for i in 0:4:31
    println(i)
    lines!([0,31], [i, i], color = :black, linewidth = 1)
    lines!([i, i], [0, 31], color = :black, linewidth = 1)
end

fig

0
4
8
12
16
20
24
28


In [98]:
pi/4

0.7853981633974483